<a href="https://colab.research.google.com/github/chahelj/AILABREAL/blob/main/ailab3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import heapq

class PuzzleNode:
    def __init__(self, board, parent=None, move="", depth=0):
        self.board = board
        self.parent = parent
        self.move = move
        self.depth = depth
        self.heuristic = self.calculate_misplaced()
        self.f_score = self.depth + self.heuristic

    def calculate_misplaced(self):
        count = 0
        goal_state = (1, 2, 3, 8, 0, 4, 7, 6, 5)
        for i in range(9):
            if self.board[i] != 0 and self.board[i] != goal_state[i]:
                count += 1
        return count

    def get_neighbors(self):
        neighbors = []
        blank_idx = self.board.index(0)
        row, col = divmod(blank_idx, 3)

        moves = [
            (-1, 0, "Up"),
            (1, 0, "Down"),
            (0, -1, "Left"),
            (0, 1, "Right")
        ]

        for dr, dc, move_name in moves:
            new_row, new_col = row + dr, col + dc
            if 0 <= new_row < 3 and 0 <= new_col < 3:
                new_blank_idx = new_row * 3 + new_col
                new_board = list(self.board)
                new_board[blank_idx], new_board[new_blank_idx] = new_board[new_blank_idx], new_board[blank_idx]
                neighbors.append(PuzzleNode(tuple(new_board), self, move_name, self.depth + 1))
        return neighbors

    def __lt__(self, other):
        return self.f_score < other.f_score


def solve_8_puzzle(start_state):
    goal_state = (1, 2, 3, 8, 0, 4, 7, 6, 5)
    open_list = []
    visited = set()

    start_node = PuzzleNode(tuple(start_state))
    heapq.heappush(open_list, start_node)

    while open_list:
        current_node = heapq.heappop(open_list)
        if current_node.board == goal_state:
            return reconstruct_path(current_node)
        if current_node.board in visited:
            continue
        visited.add(current_node.board)
        for neighbor in current_node.get_neighbors():
            if neighbor.board not in visited:
                heapq.heappush(open_list, neighbor)
    return None


def reconstruct_path(node):
    path = []
    current = node
    while current:
        path.append((current.move, current.board, current.f_score, current.heuristic))
        current = current.parent
    return path[::-1]


def print_board(board):
    for i in range(0, 9, 3):
        print(f" {board[i]}  {board[i+1]}  {board[i+2]} ")
    print("-" * 11)


if __name__ == "__main__":
    initial_puzzle = (2, 8, 3, 1, 6, 4, 0, 7, 5)
    print("chahel J")
    print("Initial Board State:")
    print_board(initial_puzzle)
    print("Goal State:")
    print_board((1, 2, 3, 8, 0, 4, 7, 6, 5))
    print("-" * 11)

    solution_path = solve_8_puzzle(initial_puzzle)

    if solution_path:
        print(f"Goal reached successfully! Solved in {len(solution_path) - 1} steps.\n")
        for step, (move, board, f_score, heuristic) in enumerate(solution_path):
            if step == 0:
                print(f"Start State: (f(n) = {f_score}, h(n) = {heuristic})")
            else:
                print(f"Step {step}: Move Blank '{move}' (f(n) = {f_score}, h(n) = {heuristic})")
            print_board(board)
    else:
        print("This initial board configuration is unsolvable.")

Initial Board State:
 2  8  3 
 1  6  4 
 0  7  5 
-----------
Goal State:
 1  2  3 
 8  0  4 
 7  6  5 
-----------
-----------
Goal reached successfully! Solved in 6 steps.

Start State: (f(n) = 5, h(n) = 5)
 2  8  3 
 1  6  4 
 0  7  5 
-----------
Step 1: Move Blank 'Right' (f(n) = 5, h(n) = 4)
 2  8  3 
 1  6  4 
 7  0  5 
-----------
Step 2: Move Blank 'Up' (f(n) = 5, h(n) = 3)
 2  8  3 
 1  0  4 
 7  6  5 
-----------
Step 3: Move Blank 'Up' (f(n) = 6, h(n) = 3)
 2  0  3 
 1  8  4 
 7  6  5 
-----------
Step 4: Move Blank 'Left' (f(n) = 6, h(n) = 2)
 0  2  3 
 1  8  4 
 7  6  5 
-----------
Step 5: Move Blank 'Down' (f(n) = 6, h(n) = 1)
 1  2  3 
 0  8  4 
 7  6  5 
-----------
Step 6: Move Blank 'Right' (f(n) = 6, h(n) = 0)
 1  2  3 
 8  0  4 
 7  6  5 
-----------


In [ ]:
from collections import deque

# Define the standard goal state as a flattened string (0 represents the empty space)
GOAL_STATE = "123456780"

def get_moves(state_str):
    """
    Finds the 0 index and returns valid next states along with the move direction.
    """
    moves = []
    idx = state_str.index('0')
    row, col = idx // 3, idx % 3

    # Movement offsets: (row_delta, col_delta, direction_name)
    directions = [(-1, 0, 'Up'), (1, 0, 'Down'), (0, -1, 'Left'), (0, 1, 'Right')]

    state_list = list(state_str)
    for dr, dc, direction in directions:
        nr, nc = row + dr, col + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            n_idx = nr * 3 + nc
            # Swap empty space with the target tile
            state_list[idx], state_list[n_idx] = state_list[n_idx], state_list[idx]
            moves.append(("".join(state_list), direction))
            # Swap back for the next iteration
            state_list[idx], state_list[n_idx] = state_list[n_idx], state_list[idx]

    return moves

def depth_limited_dfs(start_state, goal_state, limit):
    """
    Performs an explicit stack-based, depth-limited DFS.
    Returns the path of moves if found, None if limit reached without success.
    """
    # Stack stores tuples of: (current_state, path_of_moves, current_depth)
    stack = deque([(start_state, [], 0)])

    # Dictionary to track the minimum depth we reached each state at
    # to avoid exploring redundant paths deeper than previously seen.
    visited_depth = {start_state: 0}

    while stack:
        curr_state, path, depth = stack.pop()

        if curr_state == goal_state:
            return path

        if depth < limit:
            # Generate next legal configurations
            for next_state, move in get_moves(curr_state):
                # If state is unvisited, or found via a shallower path, explore it
                if next_state not in visited_depth or depth + 1 < visited_depth[next_state]:
                    visited_depth[next_state] = depth + 1
                    stack.append((next_state, path + [move], depth + 1))

    return None

def iddfs_solve(start_state, goal_state=GOAL_STATE):
    """
    Iterative Deepening DFS controller that increments depth limits progressively.
    """
    # Check if the puzzle is solvable first by inversion parity
    if not is_solvable(start_state):
        return "Unsolvable configuration"

    depth_limit = 0
    while True:
        result = depth_limited_dfs(start_state, goal_state, depth_limit)
        if result is not None:
            return result, depth_limit
        depth_limit += 1

def is_solvable(state_str):
    """
    Checks if an 8-puzzle configuration is solvable using inversion counts.
    """
    tiles = [int(x) for x in state_str if x != '0']
    inversions = 0
    for i in range(len(tiles)):
        for j in range(i + 1, len(tiles)):
            if tiles[i] > tiles[j]:
                inversions += 1
    return inversions % 2 == 0

# --- Example Execution ---
if __name__ == "__main__":
    # '0' represents the blank spot. Grid:
    # 1 2 3
    # 4 0 6
    # 7 5 8
    initial_puzzle = "123046758"

    print(f"Initial State: {initial_puzzle}")
    solution = iddfs_solve(initial_puzzle)

    if isinstance(solution, tuple):
        path, depth = solution
        print(f"Solution found at Depth Limit: {depth}")
        print(f"Moves sequence: {path}")
    else:
        print(solution)


Initial State: 123046758
Solution found at Depth Limit: 3
Moves sequence: ['Right', 'Down', 'Right']
